# 60. Target encoding the ratio block. A fold-0 probe

**Probe, no ledger row.** Fold 0 only, same treatment as `04`, `05`, `07`, `57`. Nothing here is
comparable to a five-fold row.

## The gap, and it is a real one

Two things in this repo pay, and they are the only two that ever did:

- **Target encoding**, +0.003312 at row 17, the largest feature gain in the competition.
- **The ratio block**, +0.000356 to +0.000906 across seven arms, four learner families and two
  representations, every single one positive. This repo calls it the most reproducible finding here.

**Their combination has never been run.** Checked mechanically across all eleven notebooks that
build the block: the encoder loop always iterates `COLS`, the twelve original features, and the
thirteen ratio columns are concatenated raw and never encoded. `43_ratio_block_te.ipynb` is named
for putting the block *on the encoded frame*, which is a different thing: it encodes the twelve
originals and appends raw ratios.

## Why this is the right shape to be running at all

The rank table says +0.0001 buys 15 places and +0.0005 buys 97. Adding a member to this stack
returns eight to seventy-two millionths, because a member gain reaches the stack at roughly 10:1
to 15:1 attenuation. **That attenuation is a fact about adding ONE member beside its twin. It does
not apply to a representation change, which improves every member at once.**

The repo has exactly one precedent and it is the right size: the ratio block went into six members
and **row 94 returned +0.000603**, an order of magnitude above every gate since. If encoding the
block pays at even a third of what encoding the originals paid, and it pays across families the way
the block itself did, that is the only route to +0.0005 that does not require members this repo did
not build.

## What the arms are

Numeric ratios are near-continuous, so encoding them per exact value would be noise. Each ratio
column is **binned into Q quantiles, with the bin edges fit on the training rows of the fold only**,
and the bin label is then encoded by the same nested out-of-fold encoder the twelve originals use,
fingerprinted `0642e41750ef8bab`. `31_interaction_diagnostic.ipynb` used the same quantile-bin
device at Q=12.

| arm | features | change |
|---|---|---|
| `base` | 49 | `xgb_tuned`'s frame, reproduces row 124 on fold 0 |
| `q16` | 75 | plus te and fq of thirteen ratio columns at 16 bins |
| `q32` | 75 | the same at 32 bins |

Model is `xgb_tuned`, row 124's searched configuration, held identical across arms. One variable.

## The prediction

**+0.0004 to +0.0012 on fold 0 for the better of q16 and q32.** Reasoning: `slack` is the
generator's own fingerprint and the encoder has never seen it, `screen_to_sleep` and
`weekend_ratio` are the two most plausible non-monotone responses in the block, and target encoding
is the one device that hands a tree an ordering it cannot derive by splitting.

## The case against, written first, because it has beaten the prediction ten times in twelve

**The strongest counter is row 32, and it is close to exact.** Encoding *pairs* of columns was
measured at `top9` +0.000077 and `all66` **-0.000509**, and the entry concluded that a depth-6 tree
already reaches every region a cross describes. A ratio is a function of two columns. If the tree
can already carve `daily` against `social` into rectangles, `social_share` adds nothing the tree
lacks, and encoding it adds nothing further.

The counter-counter, and the reason this is worth a kernel anyway, is that **the ratio block itself
refuted exactly that argument.** Rows 55 to 57 said crossing pays nothing because trees reach
2-way regions; the ratio block then paid on all four families. This repo's corrected rule is that
target encoding prices single-column transformations and cannot price cross-column ones, and a
ratio is precisely a cross-column transformation collapsed into one column. That rule predicts the
block pays and says nothing either way about encoding it.

**The second counter is dilution**, which is not hypothetical here. `all66` cost -0.000509 through
`colsample_bytree` alone, and this configuration samples at **0.551**, lower than the 0.8 that
produced that result. Going from 49 to 75 columns at colsample 0.551 means each tree sees 41 of 75
rather than 27 of 49, so the informative originals are competing against 26 new columns. If the
encoded ratios are weak, this arm loses for a reason that has nothing to do with ratios.

**Third: the encoder is fit on bins whose edges are themselves fit in-fold**, so there are two
fitted objects per column where the originals have one. That is more variance, and row 17's
+0.003312 came from columns with genuine repeated levels rather than manufactured ones.

If both arms land inside +/-0.0002 this is a null and the board closes for good.


In [1]:
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

PROBE_FOLD = 0
N_SPLITS = 5

# THE ONE VARIABLE: how many quantile bins the ratio columns are cut into before the
# encoder sees them. None means the arm does not encode them at all, which is row 124.
ARM_BINS = [("base", None), ("q16", 16), ("q32", 32)]
ARMS = [a for a, _ in ARM_BINS]

# Row 124's searched configuration, artifacts/oof/xgb_tuned_params.json. Held identical
# across arms so the feature set is the only thing that moves.
LR, N_EST, THREADS = 0.05, 2000, -1
XGB_TUNED = {
    "max_depth": 6,
    "min_child_weight": 38.21665211303827,
    "subsample": 0.847571294725423,
    "colsample_bytree": 0.5511988943340667,
    "reg_alpha": 0.3630617084093999,
    "reg_lambda": 2.314835250817084,
    "gamma": 0.03821923093897405,
}

ROW93_CV = 0.968005
ROW124_CV = 0.968222
ROW124_FOLD0 = None      # recomputed below from the saved vector, not quoted
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"

if SMOKE:
    N_EST = 200
    ARM_BINS = [("base", None), ("q16", 16)]
    ARMS = [a for a, _ in ARM_BINS]

print(f"SMOKE = {SMOKE}   arms {ARMS}   n_estimators {N_EST}")
print(f"probe fold {PROBE_FOLD}; the encoder still runs on the twelve originals in every arm")


SMOKE = True   arms ['base', 'q16']   n_estimators 200
probe fold 0; the encoder still runs on the twelve originals in every arm


In [2]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")

torch 2.13.0+cpu | device cpu
running locally, writing to E:\Claude\kaggle\comps\smartphone-addiction\artifacts\oof


  [ok] id is not a feature
  [ok] target is not a feature
  [ok] train and test ids do not overlap
  [ok] train and test feature lists match

  SMOKE subsamples the data, so the fold split cannot match.
  Alignment is NOT checked and this OOF is unusable.

fold alignment: FAILED, do not use this OOF
20,000 train rows, 5,000 test rows


## The encoder, fingerprinted against 13

In [3]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")

encoder fingerprint here  : 0642e41750ef8bab
encoder fingerprint in 13 : 0642e41750ef8bab
encoder: IDENTICAL to rows 17, 26 and 31's

numeric block: 9 raw + 24 encoded = 33 columns, plus 9 mask columns
row 16 had 9 numeric + 9 mask


In [4]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

1. flip all validation targets -> change in their encoding: 0.000e+00
2. flip 200 training rows -> change in their own encoding:  4.250e-03
   prior moved 4.250e-03, and these should track each other
3. flip all training targets -> change in val encoding:     9.270e-01

LEAK CHECKS: PASS


40

## The inputs, identical to row 127

In [5]:
DST, SM_, GM, WS = ("daily_screen_time_hours", "social_media_hours",
                    "gaming_hours", "work_study_hours")
SLP, WKD = "sleep_hours", "weekend_screen_time"
NOTIF, OPENS = "notifications_per_day", "app_opens_per_day"

RATIO_COLS = ["component_total", "slack", "weekend_lift", "weekend_ratio",
              "social_share", "gaming_share", "work_share", "sleep_minus_screen",
              "screen_to_sleep", "opens_per_hour", "notif_per_hour",
              "notif_per_open", "engagement"]


def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b


def ratio_block(df):
    o = pd.DataFrame(index=df.index)
    o["component_total"] = df[[SM_, GM, WS]].sum(axis=1, min_count=3)
    o["slack"] = df[DST] - o["component_total"]
    o["weekend_lift"] = df[WKD] - df[DST]
    o["weekend_ratio"] = safe_div(df[WKD], df[DST])
    o["social_share"] = safe_div(df[SM_], df[DST])
    o["gaming_share"] = safe_div(df[GM], df[DST])
    o["work_share"] = safe_div(df[WS], df[DST])
    o["sleep_minus_screen"] = df[SLP] - df[DST]
    o["screen_to_sleep"] = safe_div(df[DST], df[SLP])
    o["opens_per_hour"] = safe_div(df[OPENS], df[DST])
    o["notif_per_hour"] = safe_div(df[NOTIF], df[DST])
    o["notif_per_open"] = safe_div(df[NOTIF], df[OPENS])
    o["engagement"] = df[NOTIF] + df[OPENS]
    return o.replace([np.inf, -np.inf], np.nan).astype(np.float64)


rng = np.random.default_rng(0)
_perm = train.copy()
_perm[TARGET] = rng.permutation(train[TARGET].to_numpy())
BLOCK_OK = bool(ratio_block(_perm).equals(ratio_block(train))
                and list(ratio_block(train).columns) == RATIO_COLS)
print(f"ratio block is a pure function of the features, not of y: {BLOCK_OK}")
del _perm

RB_TR = ratio_block(train).to_numpy(np.float32)
RB_TE = ratio_block(test).to_numpy(np.float32)
print(f"row 106 fed {len(NUM_COLS) + len(ENC_COLS)} numeric + {len(NUM_COLS)} mask columns")
print(f"these arms feed {len(NUM_COLS) + len(ENC_COLS) + len(RATIO_COLS)} numeric "
      f"+ {len(NUM_COLS)} mask")

cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)
Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)

ratio block is a pure function of the features, not of y: True
row 106 fed 33 numeric + 9 mask columns
these arms feed 46 numeric + 9 mask


## The architecture

Written from the paper's description. The pieces that differ from row 127 are the periodic
embedding of every numeric feature, the learnable front scale, and the parameter groups that give
scale and bias tensors their own learning rate and weight decay.

In [6]:
import gc

from xgboost import XGBClassifier

# The bin-then-encode device. Edges are fit on the TRAINING rows of the fold only, then
# applied to validation and test, so a validation row's bin assignment never depends on
# a validation target and never on a validation quantile.
#
# This is the same quantile-bin device 31_interaction_diagnostic.ipynb used at Q=12,
# reused here so the two are comparable.


def bin_ratios(rb_tr, rb_other_list, q):
    """Quantile-bin the ratio block. Returns (train_labels, [other_labels...])."""
    out_tr = np.empty(rb_tr.shape, dtype=np.int32)
    outs = [np.empty(o.shape, dtype=np.int32) for o in rb_other_list]
    for j in range(rb_tr.shape[1]):
        col = rb_tr[:, j]
        finite = col[np.isfinite(col)]
        # Duplicate edges collapse silently and would merge bins without warning, so the
        # edge count actually used is reported rather than assumed to be q.
        edges = np.unique(np.quantile(finite, np.linspace(0, 1, q + 1)[1:-1]))
        out_tr[:, j] = np.digitize(col, edges)
        # NaN digitizes to the top bin, which would merge it with the largest values.
        # It gets its own label instead, because 4 to 20 percent of every column is missing.
        out_tr[np.isnan(col), j] = len(edges) + 1
        for k, o in enumerate(rb_other_list):
            c = o[:, j]
            outs[k][:, j] = np.digitize(c, edges)
            outs[k][np.isnan(c), j] = len(edges) + 1
    return out_tr, outs


def encode_binned(lab_tr, lab_va, lab_te, yy_tr, prior, seed=SEED):
    """Nested out-of-fold target and frequency encoding of the binned labels.

    Same protocol as `encode_fold` above: validation and test rows use statistics fit on
    the whole training portion, training rows use inner out-of-fold statistics.
    """
    n = lab_tr.shape[1]
    e_tr = np.empty((lab_tr.shape[0], 2 * n))
    e_va = np.empty((lab_va.shape[0], 2 * n))
    e_te = np.empty((lab_te.shape[0], 2 * n))
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(lab_tr))
    for j in range(n):
        mean, freq = _stats(lab_tr[:, j], yy_tr, prior)
        e_va[:, 2 * j], e_va[:, 2 * j + 1] = _apply(lab_va[:, j], mean, freq, prior)
        e_te[:, 2 * j], e_te[:, 2 * j + 1] = _apply(lab_te[:, j], mean, freq, prior)
        for itr, iva in splits:
            im, if_ = _stats(lab_tr[itr, j], yy_tr[itr], prior)
            e_tr[iva, 2 * j], e_tr[iva, 2 * j + 1] = _apply(lab_tr[iva, j], im, if_, prior)
    return e_tr, e_va, e_te


def make_xgb():
    return XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=N_EST,
        random_state=SEED, n_jobs=THREADS, verbosity=0, **XGB_TUNED)


print("bin-then-encode defined. Edges fit on training rows only; NaN gets its own label.")
print(f"model: row 124's searched XGBoost, {N_EST} trees at lr {LR}")


bin-then-encode defined. Edges fit on training rows only; NaN gets its own label.
model: row 124's searched XGBoost, 200 trees at lr 0.05


## The probe


In [7]:
LOG = OUT / "60_ratio_te.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== probe start, SMOKE={SMOKE}, fold={PROBE_FOLD}, arms={ARMS} ===")

# Row 124's fold-0 figure, recomputed from its saved vector rather than quoted, so the
# base arm has something exact to reproduce against.
_r124 = None
try:
    _v = np.load(locate("xgb_tuned_oof.npy"))
    if len(_v) == len(train):
        _r124 = float(roc_auc_score(y[folds == PROBE_FOLD], _v[folds == PROBE_FOLD]))
        note(f"row 124 fold {PROBE_FOLD} recomputed from its saved vector: {_r124:.6f}")
except FileNotFoundError:
    note("xgb_tuned_oof.npy not attached, base arm has nothing to reproduce against")
ROW124_FOLD0 = _r124


=== probe start, SMOKE=True, fold=0, arms=['base', 'q16'] ===


In [8]:
f = PROBE_FOLD
tr_i = np.where(folds != f)[0]
va_i = np.where(folds == f)[0]
prior = float(y[tr_i].mean())

# The twelve originals, encoded exactly as every other notebook encodes them. Shared by
# all arms, so it is built once.
Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
BASE_COLS = NUM_COLS + ENC_COLS
Btr = np.hstack([Etr[BASE_COLS].to_numpy(np.float32), RB_TR[tr_i]])
Bva = np.hstack([Eva[BASE_COLS].to_numpy(np.float32), RB_TR[va_i]])
Bte = np.hstack([Ete[BASE_COLS].to_numpy(np.float32), RB_TE])
CAT_TR = Etr[CAT_COLS].reset_index(drop=True)
CAT_VA = Eva[CAT_COLS].reset_index(drop=True)
note(f"base frame {Btr.shape[1]} numeric + {len(CAT_COLS)} categorical")

results = {}
t_start = time.time()
for arm, q in ARM_BINS:
    if q is None:
        Atr, Ava = Btr, Bva
    else:
        lab_tr, (lab_va, lab_te) = bin_ratios(RB_TR[tr_i], [RB_TR[va_i], RB_TE], q)
        used = [len(np.unique(lab_tr[:, j])) for j in range(lab_tr.shape[1])]
        note(f"{arm}: distinct bin labels per ratio column, min {min(used)} max {max(used)}")
        e_tr, e_va, _ = encode_binned(lab_tr, lab_va, lab_te, y[tr_i], prior)
        Atr = np.hstack([Btr, e_tr.astype(np.float32)])
        Ava = np.hstack([Bva, e_va.astype(np.float32)])

    Xtr = pd.concat([pd.DataFrame(Atr), CAT_TR], axis=1)
    Xva = pd.concat([pd.DataFrame(Ava), CAT_VA], axis=1)
    Xtr.columns = [str(c) for c in Xtr.columns]
    Xva.columns = [str(c) for c in Xva.columns]

    t0 = time.time()
    m = make_xgb().fit(Xtr, y[tr_i])
    auc = float(roc_auc_score(y[va_i], m.predict_proba(Xva)[:, 1]))
    results[arm] = {"auc": auc, "n_features": Xtr.shape[1], "bins": q}
    note(f"{arm}: fold {f} AUC {auc:.6f}  ({Xtr.shape[1]} features, "
         f"{(time.time()-t0)/60:.1f} min, total {(time.time()-t_start)/60:.1f} min)")
    del m, Xtr, Xva, Atr, Ava
    gc.collect()


base frame 46 numeric + 3 categorical


base: fold 0 AUC 0.955650  (49 features, 0.0 min, total 0.0 min)
q16: distinct bin labels per ratio column, min 17 max 17


q16: fold 0 AUC 0.956451  (75 features, 0.0 min, total 0.0 min)


In [9]:
b = results["base"]["auc"]
print(f"fold {PROBE_FOLD}, paired on identical rows, row 124's XGBoost throughout\n")
print(f"{'arm':8} {'bins':>6} {'features':>9} {'fold 0 AUC':>12} {'vs base':>11}")
for arm, q in ARM_BINS:
    r = results[arm]
    print(f"{arm:8} {str(q):>6} {r['n_features']:9d} {r['auc']:12.6f} {r['auc'] - b:+11.6f}")

if ROW124_FOLD0 is not None and not SMOKE:
    d = b - ROW124_FOLD0
    print(f"\nbase arm {b:.6f} against row 124's fold 0 of {ROW124_FOLD0:.6f}, delta {d:+.2e}")
    print("  " + ("REPRODUCED, so the arms below are measured against a live baseline"
                  if abs(d) < 5e-4 else
                  "DOES NOT REPRODUCE. Treat every number here as unexplained."))

best = max(results, key=lambda a: results[a]["auc"])
gain = results[best]["auc"] - b
print(f"\nbest arm {best}, {gain:+.6f} on one fold")
print("A fold-0 difference under about 0.0003 should not be trusted (rows 9 to 12).")
print("\nWHAT WOULD FOLLOW, so the bar is set before the number is read:")
print("  > +0.0004   worth rebuilding four families on it, which is the row 94 shape")
print("  +0.0002 to +0.0004   worth one five-fold confirmation on XGBoost alone")
print("  < +0.0002   null, and the board closes")
print("\nNo ledger row and no vector written. This prices an axis.")


fold 0, paired on identical rows, row 124's XGBoost throughout

arm        bins  features   fold 0 AUC     vs base
base       None        49     0.955650   +0.000000
q16          16        75     0.956451   +0.000801

best arm q16, +0.000801 on one fold
A fold-0 difference under about 0.0003 should not be trusted (rows 9 to 12).

WHAT WOULD FOLLOW, so the bar is set before the number is read:
  > +0.0004   worth rebuilding four families on it, which is the row 94 shape
  +0.0002 to +0.0004   worth one five-fold confirmation on XGBoost alone
  < +0.0002   null, and the board closes

No ledger row and no vector written. This prices an axis.


In [10]:
print("probe complete, nothing written")


probe complete, nothing written
